In [ ]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
alt.data_transformers.disable_max_rows()

# Data Processing Helper Functions

In [ ]:
def spliceai_mapper(df):

    df['maxSpliceAI'] = df[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

    df['splice_impact'] = '< Threshold'
    SPLICE_IMPACT_COLS = {
        'spliceAI_DS_AG': 'Acceptor Gain',
        'spliceAI_DS_AL': 'Acceptor Loss',
        'spliceAI_DS_DG': 'Donor Gain',
        'spliceAI_DS_DL': 'Donor Loss',
    }

    for col, label in SPLICE_IMPACT_COLS.items():
        df.loc[df[col] >= 0.2, 'splice_impact'] = label
    
    return df

In [ ]:
def molecular_consequence_mapper(df, remap_col):

    df = df.dropna(subset=[remap_col]).copy()
    CONSEQUENCE_EXACT = {
            'synonymous_variant': 'Synonymous',
            'intron_variant':     'Intron',
            'stop_gained':        'Stop Gained',
            'stop_lost':          'Stop Lost',
            'start_lost':         'Start Lost',
            'inframe_indel':      'Inframe Indel',
        }

    CONSEQUENCE_CONTAINS = {
        'missense': 'Missense',
        'site':     'Canonical Splice',
        'ing_var':  'Splice Region',
        'UTR':      'UTR Variant',
    }

    df[remap_col] = df[remap_col].replace(CONSEQUENCE_EXACT)
    for pattern, label in CONSEQUENCE_CONTAINS.items():
        df.loc[df[remap_col].str.contains(pattern), remap_col] = label
    

    return df

In [ ]:
raw_df = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260424_CAVASGE_SangerSGE_FindlaySGE.xlsx')

raw_df = raw_df.loc[(raw_df['ref_allele'].str.len()==1) & (raw_df['alt_allele'].str.len()==1)].copy()
print('1- Before remap: ', len(raw_df.loc[(raw_df['simplified_consequence']=='intron_variant') & (raw_df['Gene']=='VHL')]))
raw_df['pos_id']=raw_df['Gene']+':'+raw_df['hg38_start'].astype(str)+':'+raw_df['alt_allele']
raw_df['auth_reported_func_class'] = raw_df['auth_reported_func_class'].replace({
    'LOF':           'functionally_abnormal',
    'LOF1':          'functionally_abnormal',
    'LOF2':          'functionally_abnormal',
    'depleted':      'functionally_abnormal',
    'slow depleted': 'functionally_abnormal',
    'fast depleted': 'functionally_abnormal',
    'slow depleting':'functionally_abnormal',
    'fast depleting':'functionally_abnormal',
    'enriched':      'functionally_normal',
    'FUNC':          'functionally_normal',
    'Neutral':       'functionally_normal',
    'unchanged':     'functionally_normal',
    'INT':           'indeterminate',
    'Intermediate':  'indeterminate',
})
print('2- After remap: ',len(raw_df.loc[(raw_df['simplified_consequence']=='intron_variant') & (raw_df['Gene']=='VHL')]))
df = spliceai_mapper(raw_df)
df = molecular_consequence_mapper(df, 'simplified_consequence')

df.head()
print(len(df))

In [ ]:
def rna_merge_prepper(df, gene=None, pos_col='pos', alt_col='alt', threshold=None):

    if gene is not None:
        df['pos_id'] = gene + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    else:
        df['pos_id'] = df['Gene'] + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    df=df.dropna(subset=['rna_score']).copy()
    
    df['rna_consequence'] = 'normal'
    df.loc[df['rna_score'] <= threshold, 'rna_consequence'] = 'low'

    df=df[['pos_id', 'rna_score', 'rna_consequence']]

    return df

In [ ]:
vhl_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/2025_VHLBuckley.xlsx')
vhl_data = vhl_data.replace({'LOF1': 'LoF', 'LOF2':"LoF",
                             'STOP_GAINED': 'Nonsense',
                             'NON_SYNONYMOUS': 'Missense',
                             'SYNONYMOUS': 'Synonymous',
                             "CANONICAL_SPLICE":'Canonical Splice',
                             'INTRONIC': 'Intronic',
                             "SPLICE_SITE": 'Splice Region',
                             'STOP_LOST': 'Stop Lost'})

vhl_data = vhl_data.dropna(subset=['function_class']).copy()
vhl_data = rna_merge_prepper(vhl_data, gene='VHL', pos_col='hg38_pos', threshold=-3)

print(vhl_data)

In [ ]:

findlay_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260422_BRCA1_Findlay2018_wRNA.xlsx')
findlay_data = molecular_consequence_mapper(findlay_data,'simplified_consequence')

findlay_data=findlay_data.rename(columns={
                                          'mean.rna.score': 'rna_score'}
                                          )

findlay_data = rna_merge_prepper(findlay_data, gene='BRCA1', pos_col='hg38_start', alt_col='alt_allele', threshold=-2)

print(findlay_data)

In [ ]:
cava_sge=Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna')

matches=list(cava_sge.glob("*allscores*"))

all_wrna=[vhl_data, findlay_data]

rna_threshold_dict={
    'BARD1': -1.244,
    'RAD51D': -2.86
}
for match in matches:
    gene=str(match).split('/')[-1].split('.')[0]
    gene_df = pd.read_csv(match, sep ='\t')

    if len(gene_df.dropna(subset=['RNA_score'])) == 0:
        continue

    gene_df = gene_df.rename(columns={'RNA_score': 'rna_score'})
    gene_df['Gene'] = gene_df['exon'].transform(lambda x: x.split('_')[0])

    rna_threshold = rna_threshold_dict[gene]
    wrna=rna_merge_prepper(gene_df, threshold=rna_threshold)

    all_wrna.append(wrna)

cava_w_rna= pd.concat(all_wrna)

df=pd.merge(df,cava_w_rna, on='pos_id',how='left')
print('3- After RNA merging: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))

# Intial Data Processing and Z-Score Normalization

In [ ]:
normal_mask = df['auth_reported_func_class'] == 'functionally_normal'

df['z_score'] = df.groupby('Gene')['auth_reported_score'].transform(
    lambda x: (x - x[normal_mask.reindex(x.index)].mean()) 
              / x[normal_mask.reindex(x.index)].std()
)

df = df[['Gene', 'auth_reported_score', 'auth_reported_func_class', 'z_score', 
         'simplified_consequence', 'maxSpliceAI', 'splice_impact', 'rna_score', 'rna_consequence']].dropna(subset=['auth_reported_func_class'])

print('4 - After Z-scoring: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))

In [ ]:
print('5 - After Z-scoring but new cell: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))

## Z-score Normalization Sanity Check

In [ ]:
z_score_plot = alt.Chart(df).mark_boxplot().encode(
    x='Gene',
    y='z_score:Q'
).facet('auth_reported_func_class')

z_score_plot.display()

# Intron Variants, Z-score normalized

In [ ]:
# Scatter plot helper function

def corr_scatter(df, consequence, rep1, rep2, gene):
    print('6 - Right before Scatter plot: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))
    rep1_max = df[rep1].max(axis = 0)
    rep2_max = df[rep2].max(axis = 0)
    rep2_min = df[rep2].min(axis = 0)

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05
    print('6.5 - Right before Scatter plot: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))
    df = df.dropna(subset = [rep1, rep2]).copy()
    print('6.75 - Right before Scatter plot: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))
    df = df.loc[df['simplified_consequence']==consequence]

    print('7- During Scatter plot: ',len(df.loc[(df['simplified_consequence']=='Intron') & (df['Gene']=='VHL')]))
    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        color='splice_impact:N',
        tooltip = ['Gene', 'auth_reported_score']
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [1],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter+r_text).properties(title = gene).resolve_scale(x = 'shared', y = 'shared').display()

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

In [ ]:
def compute_correlations(df, rep1, rep2, consequences=None, gene_rep1='auth_reported_score'):
    if gene_rep1 is None:
        gene_rep1 = rep1

    work = df.dropna(subset=list({rep1, rep2, gene_rep1})).copy()

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) < 3:
            continue
        for gene, g in sub.groupby('Gene'):
            if len(g) >= 3:
                r, _ = stats.pearsonr(g[gene_rep1], g[rep2])
                rows.append({'consequence': cons, 'Gene': gene, 'r': r, 'n': len(g)})
        r_all, _ = stats.pearsonr(sub[rep1], sub[rep2])
        rows.append({'consequence': cons, 'Gene': 'All', 'r': r_all, 'n': len(sub)})

    return pd.DataFrame(rows)



def corr_heatmap(corr_df, rep1, rep2):
    row_order = list(corr_df['consequence'].unique())

    base = alt.Chart(corr_df).encode(
        x=alt.X('Gene:N', sort=[*corr_df.loc[corr_df['Gene'] != 'All', 'Gene'].unique(), 'All']),
        y=alt.Y('consequence:N', sort=row_order),
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('r:Q',
            scale=alt.Scale(scheme='redblue', domainMid=0, domain=[-1, 1]),
            legend=alt.Legend(title='Pearson r')
        ),
        tooltip=['Gene', 'consequence', alt.Tooltip('r:Q', format='.3f'), alt.Tooltip('n:Q', title='n')]
    )

    text = base.mark_text(fontSize=11).encode(
        text=alt.Text('r:Q', format='.2f'),
        color=alt.condition(
            alt.datum.r > 0.5, alt.value('white'), alt.value('black')
        )
    )

    return (heatmap + text).properties(title=f'Pearson r: {rep1} vs {rep2}', width = 500)


In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Intron', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')

In [ ]:
# (label, filtered_df, allowed_consequences or None for all)

consequences = ['Intron', 'Splice Region', 'Missense']

subsets = [
    ('',        df,                                                                None),
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'], None),
    ('Low RNA', df.loc[df['rna_consequence'] == 'low'],                            ['Missense']),
]

corr_df = pd.concat([
    compute_correlations(subset, 'z_score', 'maxSpliceAI',
                         consequences=[cons], gene_rep1='auth_reported_score')
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

subsets = [
    ('',     df,                                                                  None),
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'],  None),
    ('Low RNA', df.loc[(df['rna_consequence'] == 'low')],                             ['Missense']),
]

corr_df = pd.concat([
    compute_correlations(subset, 'z_score', 'maxSpliceAI',
                         consequences=[cons], gene_rep1='auth_reported_score')
      .assign(consequence=f'{label} {cons}')
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

print(corr_df)


In [ ]:
corr_heatmap(corr_df, 'z_score', 'maxSpliceAI').display()


In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Missense', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')